# Gaia star subsets for the MPI N-body project

We compare **random sampling** and **nearest-$N$ sampling** of the cleaned Gaia data. The aim is to create **reproducible inputs for MPI benchmarks**, not physically realistic stellar systems.

Place `gaia_cleaned.csv` in the **same folder** as this notebook. The 3D figures use Plotly (rotate, zoom, and pan).

## 1. Load data and calculate 3D coordinates

`RA_ICRS` and `DE_ICRS` are angles (degrees). With parallax `Plx` in milliarcseconds, distance is $d = 1000/\mathrm{Plx}$ parsecs. We only remove rows with non-positive parallax, since they cannot be converted this way.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CSV_PATH = Path("gaia_cleaned.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError("Put gaia_cleaned.csv in the same folder as this notebook.")

raw = pd.read_csv(CSV_PATH)
data = raw.loc[raw["Plx"] > 0].copy().reset_index(drop=True)
data["SpType-ELS"] = data["SpType-ELS"].str.strip()

ra = np.deg2rad(data["RA_ICRS"].to_numpy())
dec = np.deg2rad(data["DE_ICRS"].to_numpy())
distance_pc = 1000.0 / data["Plx"].to_numpy()

# Cartesian coordinates (parsecs)
data["x_pc"] = distance_pc * np.cos(dec) * np.cos(ra)
data["y_pc"] = distance_pc * np.cos(dec) * np.sin(ra)
data["z_pc"] = distance_pc * np.sin(dec)

XYZ = data[["x_pc", "y_pc", "z_pc"]].to_numpy()
print(f"Original rows: {len(raw):,}")
print(f"Valid 3D positions: {len(data):,} (excluded {len(raw) - len(data):,} non-positive parallaxes)")
data.head()

Original rows: 590,796
Valid 3D positions: 589,993 (excluded 803 non-positive parallaxes)


,Source,RA_ICRS,DE_ICRS,Plx,e_Plx,Mass-Flame,SpType-ELS,pmRA,pmDE,RV,x_pc,y_pc,z_pc
0,159577521313026816,71.255571,31.846653,0.8001,0.0554,5.764,O,1.798,-5.479,21.25,341.173382,1005.385546,659.477123
1,177599302871495040,65.147085,37.570054,0.3269,0.0532,4.783,O,-0.497,-1.593,-62.88,1019.044396,2200.075655,1865.191159
2,177812440625769856,64.818613,38.061037,0.4897,0.0330,5.820,O,1.832,0.046,47.31,684.108162,1455.030570,1258.935243
3,178300417628833664,66.653721,37.546868,0.8147,0.0294,4.000,O,23.448,-12.353,-80.53,385.661411,893.508476,748.017907
4,181446292196113920,78.389326,32.517360,0.1755,0.0506,4.315,O,0.630,1.582,-14.13,967.000178,4706.406109,3062.992186


## 2. Select stars using two methods

- **Random:** a fixed seed gives the same stars every time.
- **Nearest:** select the $N$ stars closest to one fixed centre (the median 3D position).

For both methods, smaller samples are **nested** inside larger ones.

In [2]:
SEED = 42
N_VALUES = [1_000, 5_000, 20_000]  # Change these as needed

# Build the orderings once; reuse them for all N.
rng = np.random.default_rng(SEED)
random_order = rng.permutation(len(data))

CENTER = np.median(XYZ, axis=0)  # fixed 3D reference point (parsecs)
distance_sq = np.sum((XYZ - CENTER) ** 2, axis=1)
nearest_order = np.argsort(distance_sq, kind="stable")


def select_stars(N, method="random"):
    """Return N stars from the cleaned data, using a reproducible ordering."""
    if not 1 <= N <= len(data):
        raise ValueError(f"N must be between 1 and {len(data):,}")
    if method == "random":
        indices = random_order[:N]
    elif method == "nearest":
        indices = nearest_order[:N]
    else:
        raise ValueError("method must be 'random' or 'nearest'")
    return data.iloc[indices].copy()


random_1000 = select_stars(1_000, "random")
nearest_1000 = select_stars(1_000, "nearest")
print(f"Random N=1000: {len(random_1000)} stars")
print(f"Nearest N=1000: {len(nearest_1000)} stars")
print("Fixed selection centre (pc):", np.round(CENTER, 2))

Random N=1000: 1000 stars
Nearest N=1000: 1000 stars
Fixed selection centre (pc): [396.28 447.53 464.04]


## 3. Check the selections

The distance summary shows how widely spread each selection is. Both methods have **exactly the same number of stars** for a given $N$.

In [3]:
summary = []
for N in N_VALUES:
    for method in ("random", "nearest"):
        subset = select_stars(N, method)
        coordinates = subset[["x_pc", "y_pc", "z_pc"]].to_numpy()
        radius_pc = np.linalg.norm(coordinates - CENTER, axis=1)
        summary.append({
            "N": N,
            "method": method,
            "median distance to centre (pc)": round(np.median(radius_pc)),
            "95th percentile distance (pc)": round(np.percentile(radius_pc, 95)),
        })

pd.DataFrame(summary)

,N,method,median distance to centre (pc),95th percentile distance (pc)
0,1000,random,755,7435
1,1000,nearest,48,60
2,5000,random,767,8512
3,5000,nearest,85,110
4,20000,random,755,8518
5,20000,nearest,148,186


## 4. Interactive 3D visualisation

Each figure compares **random** and **nearest** selections for the same $N$. Each panel auto-zooms independently, so compare the spatial scales as well as the shapes.

Because the dataset contains some extremely distant stars, the *random plot only* hides its most distant 1% by default. **This affects the plot, not the selected dataset.** Change `random_plot_percentile=100` to display every selected star.

In [4]:
def compare_subsets(N, random_plot_percentile=99):
    """Show interactive 3D random vs. nearest-N scatter plots."""
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=(f"Random sample (N={N:,})", f"Nearest sample (N={N:,})"),
        horizontal_spacing=0.04,
    )

    for column, method, color in [
        (1, "random", "#51B8E8"),
        (2, "nearest", "#F19C52"),
    ]:
        subset = select_stars(N, method)
        xyz = subset[["x_pc", "y_pc", "z_pc"]].to_numpy()

        if method == "random" and random_plot_percentile < 100:
            radial_distance = np.linalg.norm(xyz - CENTER, axis=1)
            limit = np.percentile(radial_distance, random_plot_percentile)
            xyz = xyz[radial_distance <= limit]

        fig.add_trace(
            go.Scatter3d(
                x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
                mode="markers",
                marker=dict(size=1.8, color=color, opacity=0.7),
                name=method,
                hovertemplate="x: %{x:.0f} pc<br>y: %{y:.0f} pc<br>z: %{z:.0f} pc<extra></extra>",
                showlegend=False,
            ),
            row=1, col=column,
        )

    fig.update_scenes(
        xaxis_title="x (pc)", yaxis_title="y (pc)", zaxis_title="z (pc)",
        aspectmode="data",
    )
    fig.update_layout(
        template="plotly_dark", height=540,
        margin=dict(l=0, r=0, t=65, b=0),
        title=f"Gaia selection methods — {N:,} stars each",
    )
    fig.show()


# One interactive comparison per sample size
for N in N_VALUES:
    compare_subsets(N)

## 5. Export subsets (optional)

Set `SAVE_SUBSETS=True` to save CSVs for the serial and MPI implementations. They include all original columns plus `x_pc`, `y_pc`, and `z_pc`. The same files should be reused across parallel benchmarks.

In [5]:
SAVE_SUBSETS = False

if SAVE_SUBSETS:
    out = Path("gaia_subsets")
    out.mkdir(exist_ok=True)
    for N in N_VALUES:
        for method in ("random", "nearest"):
            filename = out / f"gaia_{method}_{N}.csv"
            select_stars(N, method).to_csv(filename, index=False)
            print("Saved:", filename)

# Example for a single dataset:
# select_stars(5_000, "random").to_csv("gaia_random_5000.csv", index=False)

### MPI benchmark reminder

For **strong scaling**, keep the **same selected stars and $N$** and vary the MPI rank count. For direct summation, work per time step is $O(N^2)$ regardless of sampling method. **Numerical validation** should compare the MPI result with the serial result for identical initial conditions.